# Reversible repair GPU compatibility and timing check

One proposed T4 check,120controlled seconds maximum; setup/upload/export/idle extra.
Two updates on one TRAIN cube5 example plus restore/replay of update2: three
optimizer steps total. Exact trace,start,state and full checkpoint payload must
match. Includes a separate device deletion/rebirth probe,one untrained32-step
positive-growth timing probe for CGR1 and RGR1,and one captured learned rollout.
Timing includes CPU connectivity flood fill and device transfers. This is not
quality evaluation or a512-update paired experiment. No automatic continuation.
Expected T4,Python3.13.15,Torch2.11.0+cu130,NumPy2.1.3,CUDA13.0,cuDNN92700.
Changed runtime stops; deterministic algorithms remain enabled.

Open the supplied notebook; upload this package only. Leave APPROVED_SEED_JOB=False
until this exact120s check is approved,then set True and execute once. Download
full ZIP and receipt,including failures; return both. No Drive access or retry.
Do not execute other historical runners included as source dependencies.
Original CGR1 and prior experiments remain unchanged. Reversible update semantics
and CPU recovery are locally tested; GPU correctness and timing remain unverified.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, zipfile, json, uuid, subprocess, sys
EXPECTED_SHA256 = '970ae29e8b0dad9bf8dff79563e9ca5d68bbbe171e1b49cc0ac17536fcec640a'
ARCHIVE_NAME = 'NCA-RGR1-cu130-Preflight-Package.zip'
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Select exactly the supplied CGR3 ZIP')
raw = next(iter(uploaded.values()))
if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256: raise ValueError('Package checksum differs')
source = Path('/content') / ('cgr3-compat-upload-' + uuid.uuid4().hex + '.zip')
with source.open('xb') as f: f.write(raw)
PACKAGE = Path('/content') / ('nca-rgr1-compat-' + uuid.uuid4().hex)
with zipfile.ZipFile(source) as z:
    manifest = json.loads(z.read('manifest.json'))
    names = z.namelist()
    if len(names) != len(set(names)) or set(names) != set(manifest['files']) | {'manifest.json'}: raise ValueError('Unexpected members')
    if sum(x.file_size for x in z.infolist()) > 100_000_000: raise ValueError('Expanded size limit')
    for name in names:
        p = PurePosixPath(name)
        if p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or str(p) != name: raise ValueError('Unsafe path')
        if ((z.getinfo(name).external_attr >> 16) & 0o170000) == 0o120000: raise ValueError('Symlink member')
        if name != 'manifest.json' and hashlib.sha256(z.read(name)).hexdigest() != manifest['files'][name]: raise ValueError('Member hash differs')
    PACKAGE.mkdir()
    for name in names:
        target = PACKAGE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open('xb') as f: f.write(z.read(name))
subprocess.run([sys.executable, '-c', 'from nca.reversible_package import verify; verify(".")'], cwd=PACKAGE, check=True)
print('Package verified:', PACKAGE)


In [ ]:
MODEL_SEED = 1201  # Only seed1201 is proposed.
APPROVED_SEED_JOB = False
if not APPROVED_SEED_JOB: raise RuntimeError('Stop until this seed job and backup plan are approved')
result = subprocess.run([sys.executable, str(PACKAGE / 'scripts/colab_reversible_compatibility.py'),
    '--seed', str(MODEL_SEED), '--device', 'cuda:0', '--approved-seed-job', '--seconds', '120'], cwd=PACKAGE)
print('Exit code:', result.returncode, '- download evidence next, including failures.')


In [ ]:
exports = sorted((PACKAGE / 'compatibility-runs').glob('*.zip'))
if not exports: raise RuntimeError('No export. Preserve runtime and send the error; do not rerun.')
for archive in exports:
    files.download(str(archive))
    files.download(str(archive.with_suffix('.receipt.json')))
print('Return ZIP and receipt for local verification; disconnect idle GPU after verified download.')
print('No additional seed or automatic retry is planned.')
